# Logistic Regression vs XGBoost Comparison

Train on 2003-2021, predict 2022-2025 (same eval as submission.ipynb).

Models compared:
1. XGBoost (current blended model)
2. Logistic Regression — all features
3. Logistic Regression — minimal features (Elo, Seed, PPG, Win%)
4. Pure Elo baseline
5. Blends of LR + Elo

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.data_loader import load_men_data, load_women_data
from src.feature_engineering import *
from src.model import train_final_model, LEAN_PARAMS, LEAN_FEATURES, brier_score
from src.model_logreg import (
    train_logreg, predict_logreg, logreg_holdout_eval,
    print_logreg_coefficients, MINIMAL_FEATURES, MINIMAL_FEATURES_MEN
)

## Load Data & Build Features

In [2]:
data_dir = f'{project_root}/data'

# Men
m_data = load_men_data(data_dir)
m_elo = compute_elo_ratings(m_data['MComSsn'])
m_seeds = parse_seeds(m_data['MTrnySeeds'])
m_stats = compute_season_stats(m_data['MDetSsn'])
massey = compute_massey_features(m_data['MOrdinals'])
m_conf = compute_conference_strength(m_data['MConf'], m_elo)
m_features = build_team_features(m_elo, m_seeds, m_stats, massey, m_conf)

# Women
w_data = load_women_data(data_dir)
w_elo = compute_elo_ratings(w_data['WComSsn'])
w_seeds = parse_seeds(w_data['WTrnySeeds'])
w_stats = compute_season_stats(w_data['WDetSsn'])
w_conf = compute_conference_strength(w_data['WConf'], w_elo)
w_features = build_team_features(w_elo, w_seeds, stats_df=w_stats, conf_df=w_conf)

print('Features loaded.')

Features loaded.


In [3]:
# Build matchup matrices and split
m_matchups = build_matchup_matrix(m_data['MDetTrny'], m_features)
m_train = m_matchups[m_matchups['Season'] < 2022]
m_test = m_matchups[m_matchups['Season'] >= 2022]

w_matchups = build_matchup_matrix(w_data['WDetTrny'], w_features)
w_train = w_matchups[w_matchups['Season'] < 2022]
w_test = w_matchups[w_matchups['Season'] >= 2022]

print(f'Men: {len(m_train)} train, {len(m_test)} test games')
print(f'Women: {len(w_train)} train, {len(w_test)} test games')

Men: 1181 train, 268 test games
Women: 693 train, 268 test games


In [4]:
# Compute Elo probabilities for holdout sets
def get_elo_probs(test_df, elo_df):
    elo_a = test_df.merge(elo_df, left_on=['Season', 'TeamA'], right_on=['Season', 'TeamID'])['Elo']
    elo_b = test_df.merge(elo_df, left_on=['Season', 'TeamB'], right_on=['Season', 'TeamID'])['Elo']
    return elo_to_prob(elo_a.values, elo_b.values)

m_elo_probs = get_elo_probs(m_test, m_elo)
w_elo_probs = get_elo_probs(w_test, w_elo)

print(f'Men Elo baseline:   {brier_score(m_test["Target"].values, m_elo_probs):.5f}')
print(f'Women Elo baseline: {brier_score(w_test["Target"].values, w_elo_probs):.5f}')

Men Elo baseline:   0.20259
Women Elo baseline: 0.15853


## 1. XGBoost Baseline (current model)

In [5]:
# Men XGBoost
MEN_FEAT = [c for c in LEAN_FEATURES if c in m_train.columns]
m_xgb_model, _ = train_final_model(m_train, feature_cols=MEN_FEAT, xgb_params=LEAN_PARAMS, num_boost_round=200)
m_xgb_preds = m_xgb_model.predict(xgb.DMatrix(m_test[MEN_FEAT]))

WOMEN_FEAT = [
    'Elo_diff', 'SeedNum_diff', 'eFG_off_diff', 'TO_rate_off_diff', 'OR_pct_diff', 'FT_rate_off_diff',
    'eFG_def_diff', 'TO_rate_def_diff', 'DR_pct_diff', 'FT_rate_def_diff',
    'Tempo_diff', 'PPG_diff', 'PPG_allowed_diff', 'Win_pct_diff', 'Conf_Elo_mean_diff',
]
WOMEN_FEAT = [c for c in WOMEN_FEAT if c in w_train.columns]
w_xgb_model, _ = train_final_model(w_train, feature_cols=WOMEN_FEAT, xgb_params=LEAN_PARAMS, num_boost_round=200)
w_xgb_preds = w_xgb_model.predict(xgb.DMatrix(w_test[WOMEN_FEAT]))

m_xgb_blend = 0.55 * m_xgb_preds + 0.45 * m_elo_probs
w_xgb_blend = 0.60 * w_xgb_preds + 0.40 * w_elo_probs

print('=== XGBoost Results ===')
print(f'Men XGB only:    {brier_score(m_test["Target"].values, m_xgb_preds):.5f}')
print(f'Men XGB+Elo:     {brier_score(m_test["Target"].values, m_xgb_blend):.5f}')
print(f'Women XGB only:  {brier_score(w_test["Target"].values, w_xgb_preds):.5f}')
print(f'Women XGB+Elo:   {brier_score(w_test["Target"].values, w_xgb_blend):.5f}')

=== XGBoost Results ===
Men XGB only:    0.19943
Men XGB+Elo:     0.19551
Women XGB only:  0.14806
Women XGB+Elo:   0.14390


## 2. Logistic Regression — All Features

In [6]:
# Men LR — all features (same as XGBoost feature set)
print('--- Men LR (all features) ---')
m_lr_preds_all, m_lr_model_all, m_lr_scaler_all, _ = logreg_holdout_eval(
    m_matchups, feature_cols=MEN_FEAT, C=1.0, train_cutoff=2022
)
print_logreg_coefficients(m_lr_model_all, MEN_FEAT)

print()

# Women LR — all features
print('--- Women LR (all features) ---')
w_lr_preds_all, w_lr_model_all, w_lr_scaler_all, _ = logreg_holdout_eval(
    w_matchups, feature_cols=WOMEN_FEAT, C=1.0, train_cutoff=2022
)

--- Men LR (all features) ---
Holdout Brier (LogReg, C=1.0): 0.20256  (n=268)
Logistic Regression Coefficients:
  PPG_diff                       +0.7754
  Elo_diff                       +0.5800
  PPG_allowed_diff               -0.4145
  Massey_median_diff             -0.3466
  Win_pct_diff                   -0.3391
  Tempo_diff                     -0.3389
  SeedNum_diff                   -0.3378
  TO_rate_def_diff               +0.1618
  Rank_POM_diff                  -0.1590
  FT_rate_off_diff               -0.1287
  DR_pct_diff                    -0.1152
  eFG_off_diff                   -0.1053
  FT_rate_def_diff               -0.0476
  OR_pct_diff                    +0.0430
  eFG_def_diff                   +0.0381
  TO_rate_off_diff               +0.0247
  Conf_Elo_mean_diff             -0.0173

--- Women LR (all features) ---
Holdout Brier (LogReg, C=1.0): 0.14285  (n=268)


In [7]:
# Blend LR all features + Elo
print('--- LR (all features) + Elo blends ---')
for w in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]:
    m_blend = w * m_lr_preds_all + (1 - w) * m_elo_probs
    w_blend = w * w_lr_preds_all + (1 - w) * w_elo_probs
    m_bs = brier_score(m_test['Target'].values, m_blend)
    w_bs = brier_score(w_test['Target'].values, w_blend)
    print(f'w={w:.1f}: Men={m_bs:.5f}, Women={w_bs:.5f}')

--- LR (all features) + Elo blends ---
w=0.3: Men=0.19954, Women=0.14822
w=0.4: Men=0.19910, Women=0.14586
w=0.5: Men=0.19895, Women=0.14402
w=0.6: Men=0.19909, Women=0.14272
w=0.7: Men=0.19952, Women=0.14195
w=0.8: Men=0.20025, Women=0.14172
w=1.0: Men=0.20256, Women=0.14285


## 3. Logistic Regression — Minimal Features

In [8]:
# Men LR — minimal features (with Pomeroy)
MEN_MINIMAL = [c for c in MINIMAL_FEATURES_MEN if c in m_train.columns]
print(f'Men minimal features: {MEN_MINIMAL}')
print('--- Men LR (minimal) ---')
m_lr_preds_min, m_lr_model_min, m_lr_scaler_min, _ = logreg_holdout_eval(
    m_matchups, feature_cols=MEN_MINIMAL, C=1.0, train_cutoff=2022
)
print_logreg_coefficients(m_lr_model_min, MEN_MINIMAL)

print()

# Women LR — minimal features
WOMEN_MINIMAL = [c for c in MINIMAL_FEATURES if c in w_train.columns]
print(f'Women minimal features: {WOMEN_MINIMAL}')
print('--- Women LR (minimal) ---')
w_lr_preds_min, w_lr_model_min, w_lr_scaler_min, _ = logreg_holdout_eval(
    w_matchups, feature_cols=WOMEN_MINIMAL, C=1.0, train_cutoff=2022
)
print_logreg_coefficients(w_lr_model_min, WOMEN_MINIMAL)

Men minimal features: ['Elo_diff', 'SeedNum_diff', 'Rank_POM_diff']
--- Men LR (minimal) ---
Holdout Brier (LogReg, C=1.0): 0.19504  (n=268)
Logistic Regression Coefficients:
  Rank_POM_diff                  -0.7117
  Elo_diff                       +0.6167
  SeedNum_diff                   -0.2578

Women minimal features: ['Elo_diff', 'SeedNum_diff']
--- Women LR (minimal) ---
Holdout Brier (LogReg, C=1.0): 0.14030  (n=268)
Logistic Regression Coefficients:
  SeedNum_diff                   -1.2210
  Elo_diff                       +1.0334


In [9]:
# Blend LR minimal + Elo
print('--- LR (minimal) + Elo blends ---')
for w in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]:
    m_blend = w * m_lr_preds_min + (1 - w) * m_elo_probs
    w_blend = w * w_lr_preds_min + (1 - w) * w_elo_probs
    m_bs = brier_score(m_test['Target'].values, m_blend)
    w_bs = brier_score(w_test['Target'].values, w_blend)
    print(f'w={w:.1f}: Men={m_bs:.5f}, Women={w_bs:.5f}')

--- LR (minimal) + Elo blends ---
w=0.3: Men=0.19857, Women=0.15011
w=0.4: Men=0.19757, Women=0.14787
w=0.5: Men=0.19673, Women=0.14590
w=0.6: Men=0.19606, Women=0.14422
w=0.7: Men=0.19555, Women=0.14282
w=0.8: Men=0.19521, Women=0.14170
w=1.0: Men=0.19504, Women=0.14030


## 4. Regularization Sweep for LR

In [10]:
# Test different regularization strengths
print('--- Regularization sweep (all features) ---')
print(f'{"C":>8s}  {"Men Brier":>10s}  {"Women Brier":>12s}')
for C in [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]:
    m_preds, _, _, _ = logreg_holdout_eval(m_matchups, feature_cols=MEN_FEAT, C=C, train_cutoff=2022)
    w_preds, _, _, _ = logreg_holdout_eval(w_matchups, feature_cols=WOMEN_FEAT, C=C, train_cutoff=2022)
    m_bs = brier_score(m_test['Target'].values, m_preds)
    w_bs = brier_score(w_test['Target'].values, w_preds)
    print(f'{C:>8.2f}  {m_bs:>10.5f}  {w_bs:>12.5f}')

--- Regularization sweep (all features) ---
       C   Men Brier   Women Brier
Holdout Brier (LogReg, C=0.01): 0.19925  (n=268)
Holdout Brier (LogReg, C=0.01): 0.14758  (n=268)
    0.01     0.19925       0.14758
Holdout Brier (LogReg, C=0.05): 0.20108  (n=268)
Holdout Brier (LogReg, C=0.05): 0.14148  (n=268)
    0.05     0.20108       0.14148
Holdout Brier (LogReg, C=0.1): 0.20169  (n=268)
Holdout Brier (LogReg, C=0.1): 0.14132  (n=268)
    0.10     0.20169       0.14132
Holdout Brier (LogReg, C=0.5): 0.20242  (n=268)
Holdout Brier (LogReg, C=0.5): 0.14232  (n=268)
    0.50     0.20242       0.14232
Holdout Brier (LogReg, C=1.0): 0.20256  (n=268)
Holdout Brier (LogReg, C=1.0): 0.14285  (n=268)
    1.00     0.20256       0.14285
Holdout Brier (LogReg, C=5.0): 0.20245  (n=268)
Holdout Brier (LogReg, C=5.0): 0.14363  (n=268)
    5.00     0.20245       0.14363
Holdout Brier (LogReg, C=10.0): 0.20247  (n=268)
Holdout Brier (LogReg, C=10.0): 0.14382  (n=268)
   10.00     0.20247       0.1438

## 5. Summary Comparison

In [11]:
# Collect all results
m_y = m_test['Target'].values
w_y = w_test['Target'].values

results = {
    'Pure Elo': (brier_score(m_y, m_elo_probs), brier_score(w_y, w_elo_probs)),
    'XGB only': (brier_score(m_y, m_xgb_preds), brier_score(w_y, w_xgb_preds)),
    'XGB+Elo blend': (brier_score(m_y, m_xgb_blend), brier_score(w_y, w_xgb_blend)),
    'LR all features': (brier_score(m_y, m_lr_preds_all), brier_score(w_y, w_lr_preds_all)),
    'LR minimal': (brier_score(m_y, m_lr_preds_min), brier_score(w_y, w_lr_preds_min)),
}

# Also compute best LR+Elo blends
for label, m_lr, w_lr in [
    ('LR all + Elo', m_lr_preds_all, w_lr_preds_all),
    ('LR min + Elo', m_lr_preds_min, w_lr_preds_min),
]:
    best_m, best_w_score = 999, 999
    for wt in np.arange(0.0, 1.05, 0.05):
        m_b = brier_score(m_y, wt * m_lr + (1 - wt) * m_elo_probs)
        w_b = brier_score(w_y, wt * w_lr + (1 - wt) * w_elo_probs)
        if m_b < best_m: best_m = m_b
        if w_b < best_w_score: best_w_score = w_b
    results[label] = (best_m, best_w_score)

print(f'{"Model":25s}  {"Men Brier":>10s}  {"Women Brier":>12s}')
print('-' * 52)
for name, (m_bs, w_bs) in results.items():
    print(f'{name:25s}  {m_bs:>10.5f}  {w_bs:>12.5f}')

Model                       Men Brier   Women Brier
----------------------------------------------------
Pure Elo                      0.20259       0.15853
XGB only                      0.19943       0.14806
XGB+Elo blend                 0.19551       0.14390
LR all features               0.20256       0.14285
LR minimal                    0.19504       0.14030
LR all + Elo                  0.19895       0.14172
LR min + Elo                  0.19502       0.14030
